[Reference](https://python.plainenglish.io/i-fixed-my-python-logging-with-3-tools-here-is-how-b6f45e613083$0)

# Why Default Python Logging Was Not Enough

In [5]:
import logging

logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)
logger.info("User login attempt")
logger.error("Database connection failed")

ERROR:__main__:Database connection failed


# Tool 1: Loguru


In [6]:
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 828.2 kB/s eta 0:00:00


In [7]:
from loguru import logger

logger.info("User login attempt")
logger.warning("Rate limit approaching for user {user_id}", user_id=123)
logger.error("Database connection failed")
logger.debug("Query executed in {time}ms", time=45)

2026-08-11 14:12:11.409 | INFO     | __main__:<cell line: 0>:3 - User login attempt
2026-08-11 14:12:11.411 | WARNING  | __main__:<cell line: 0>:4 - Rate limit approaching for user 123
2026-08-11 14:12:11.413 | ERROR    | __main__:<cell line: 0>:5 - Database connection failed
2026-08-11 14:12:11.414 | DEBUG    | __main__:<cell line: 0>:6 - Query executed in 45ms


In [8]:
from loguru import logger
import sys

logger.remove()
logger.add(
    sys.stdout,
    format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {name}:{function}:{line} - {message}",
    level="DEBUG",
    colorize=True
)
logger.add(
    "logs/app_{time:YYYY-MM-DD}.log",
    rotation="100 MB",
    retention="30 days",
    compression="zip",
    level="INFO",
    format="{time} | {level} | {name}:{function}:{line} | {message}"
)
logger.add(
    "logs/errors_{time:YYYY-MM-DD}.log",
    rotation="50 MB",
    retention="60 days",
    level="ERROR",
    backtrace=True,
    diagnose=True
)

3

# Tool 2: Structlog


In [9]:
!pip install structlog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 920.8 kB/s eta 0:00:00


In [10]:
import structlog

structlog.configure(
    processors=[
        structlog.stdlib.filter_by_level,
        structlog.stdlib.add_logger_name,
        structlog.stdlib.add_log_level,
        structlog.stdlib.PositionalArgumentsFormatter(),
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.StackInfoRenderer(),
        structlog.processors.format_exc_info,
        structlog.processors.JSONRenderer()
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
    wrapper_class=structlog.stdlib.BoundLogger,
    cache_logger_on_first_use=True,
)
logger = structlog.get_logger()

In [11]:
import structlog
import uuid

logger = structlog.get_logger()
def handle_request(user_id: int, endpoint: str):
    request_id = str(uuid.uuid4())
    log = logger.bind(
        request_id=request_id,
        user_id=user_id,
        endpoint=endpoint
    )
    log.info("request_started")
    try:
        result = process_request(user_id)
        log.info("request_completed", status="success", result_count=len(result))
        return result
    except Exception as e:
        log.error("request_failed", error=str(e), error_type=type(e).__name__)
        raise

# Tool 3: Sentry

In [12]:
!pip install sentry_sdk

In [13]:
import sentry_sdk
from sentry_sdk.integrations.fastapi import FastApiIntegration
from sentry_sdk.integrations.sqlalchemy import SqlalchemyIntegration
import os

sentry_sdk.init(
    dsn=os.environ.get("SENTRY_DSN"),
    integrations=[
        FastApiIntegration(transaction_style="endpoint"),
        SqlalchemyIntegration(),
    ],
    traces_sample_rate=0.2,
    profiles_sample_rate=0.1,
    environment=os.environ.get("ENVIRONMENT", "production"),
    release=os.environ.get("APP_VERSION", "unknown"),
)

In [14]:
import sentry_sdk
from loguru import logger

def process_payment(user_id: int, amount: float):
    try:
        result = payment_gateway.charge(user_id, amount)
        return result
    except PaymentGatewayError as e:
        logger.error(f"Payment failed for user {user_id}: {str(e)}")
        sentry_sdk.capture_exception(e)
        raise